# sensor_load_backfill

**Purpose:** Load the PI-style simulated sensor backfill (DEC-032) into Bronze, unchanged, with lineage columns.
All readings are simulated by `scripts/sensor_backfill.py`; this is not AVEVA PI System.

**Input:** `Files/config/sensor.json`; `Files/raw/sensor/backfill_<month>.parquet` and
`Files/raw/sensor/sensor_asset_hierarchy.parquet` (generated locally, uploaded with `fab cp`).

**Output tables:**
- `sensor_asset_hierarchy` (overwrite): facility -> well -> tag nodes with AF style paths, units and ranges
- `bronze_sensor_backfill` (reload by file with `replaceWhere`): `tag_name`, `ts_utc` (UTC), `value`, `source`
- `sensor_load_rejects` (append, keyed by `run_id`): rows that fail a key check, with the reason
- `pipeline_run_log` (append): one row per loaded file

**Checks before anything is written:** keys not null, `(tag_name, ts_utc)` unique, every timestamp inside the local
month, every tag in the hierarchy, every hierarchy well in `gold_dim_well`, unique AF paths and tag names, the
expected number of tags per well and of hours (721 for 2025-11). Failed rows go to `sensor_load_rejects` and the
notebook raises; Bronze is then left as it was. After the write, rows in = rows out.

Run on its own after generating a new backfill; not part of a pipeline.


In [ ]:
import json, uuid
from datetime import datetime
from zoneinfo import ZoneInfo
from pyspark.sql import functions as F
from pyspark.sql.window import Window

RUN_ID = str(uuid.uuid4())[:8]
with open("/lakehouse/default/Files/config/sensor.json") as fh:
    CFG = json.load(fh)
MONTH = CFG["month"]["production_month"]
SRC = "Files/raw/sensor"
READINGS_FILE = f"backfill_{MONTH}.parquet"
HIERARCHY_FILE = "sensor_asset_hierarchy.parquet"
TAGS_PER_WELL = 5

year, month = (int(x) for x in MONTH.split("-"))
tz = ZoneInfo(CFG["month"]["timezone"])
start_utc = datetime(year, month, 1, tzinfo=tz)
end_utc = datetime(year + month // 12, month % 12 + 1, 1, tzinfo=tz)
print("RUN_ID:", RUN_ID, "| month", MONTH, "| UTC", start_utc.astimezone(ZoneInfo("UTC")), "to", end_utc.astimezone(ZoneInfo("UTC")))

t0 = datetime.now()
hierarchy = spark.read.parquet(f"{SRC}/{HIERARCHY_FILE}")
raw = spark.read.parquet(f"{SRC}/{READINGS_FILE}")
rows_in = raw.count()
tags = hierarchy.filter(F.col("node_type") == "tag")

# Row checks on the readings: failed rows go to sensor_load_rejects
checked = (raw.join(tags.select("tag_name", F.lit(True).alias("_known_tag")), "tag_name", "left")
    .withColumn("_key_count", F.count(F.lit(1)).over(Window.partitionBy("tag_name", "ts_utc"))))
reason = (F.when(F.col("tag_name").isNull() | F.col("ts_utc").isNull(), "null_key")
           .when((F.col("ts_utc") < F.lit(start_utc)) | (F.col("ts_utc") >= F.lit(end_utc)), "outside_month")
           .when(F.col("_known_tag").isNull(), "unknown_tag")
           .when(F.col("_key_count") > 1, "duplicate_key"))
rejects = (checked.withColumn("reject_reason", reason).filter(F.col("reject_reason").isNotNull())
    .select("tag_name", "ts_utc", "value", "source", "reject_reason",
            F.lit(READINGS_FILE).alias("_source_file"), F.lit(RUN_ID).alias("run_id")))
n_rejects = rejects.count()

# Table checks on the hierarchy and the readings as a whole
well_keys = hierarchy.filter(F.col("node_type") == "well").select("well_key")
problems = []
if hierarchy.count() != hierarchy.select("af_path").distinct().count():
    problems.append("af_path not unique")
if tags.count() != tags.select("tag_name").distinct().count():
    problems.append("tag_name not unique")
bad_tag_counts = tags.groupBy("well_key").count().filter(F.col("count") != TAGS_PER_WELL).count()
if bad_tag_counts:
    problems.append(f"{bad_tag_counts} wells without {TAGS_PER_WELL} tags")
orphans = well_keys.join(spark.table("gold_dim_well").select("well_key"), "well_key", "left_anti").count()
if orphans:
    problems.append(f"{orphans} hierarchy wells not in gold_dim_well")
n_hours = raw.select("ts_utc").distinct().count()
if n_hours != CFG["month"]["expected_hours"]:
    problems.append(f"{n_hours} distinct hours, expected {CFG['month']['expected_hours']}")
n_tags = raw.select("tag_name").distinct().count()
if n_tags != tags.count():
    problems.append(f"{n_tags} tags with readings, {tags.count()} in the hierarchy")

LOG_SCHEMA = ("run_id string, layer string, source string, file string, rows_in long, rows_out long, "
              "status string, message string, started_at timestamp, ended_at timestamp")

def log_run(file, rows_in, rows_out, status, msg):
    (spark.createDataFrame([(RUN_ID, "bronze", "sensor_backfill", file, rows_in, rows_out, status, msg, t0, datetime.now())],
                           LOG_SCHEMA).write.format("delta").mode("append").saveAsTable("pipeline_run_log"))

if n_rejects or problems:
    if n_rejects:
        rejects.write.format("delta").mode("append").saveAsTable("sensor_load_rejects")
    msg = f"{n_rejects} rejected rows; {problems}"
    log_run(READINGS_FILE, rows_in, None, "FAILED", msg[:500])
    raise ValueError(f"sensor backfill {RUN_ID} not loaded: {msg}. See sensor_load_rejects. Bronze is unchanged.")

# Write: hierarchy overwrite, readings replace this file's rows in one Delta commit
(hierarchy.withColumn("_source_file", F.lit(HIERARCHY_FILE)).withColumn("_run_id", F.lit(RUN_ID))
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("sensor_asset_hierarchy"))
bronze = (raw.withColumn("_source_file", F.lit(READINGS_FILE)).withColumn("_ingest_time", F.current_timestamp())
             .withColumn("_run_id", F.lit(RUN_ID)))
writer = bronze.write.format("delta")
if spark.catalog.tableExists("bronze_sensor_backfill"):
    writer.mode("overwrite").option("replaceWhere", f"_source_file = '{READINGS_FILE}'").saveAsTable("bronze_sensor_backfill")
else:
    writer.mode("append").saveAsTable("bronze_sensor_backfill")

rows_out = spark.table("bronze_sensor_backfill").filter(F.col("_source_file") == READINGS_FILE).count()
status = "SUCCESS" if rows_out == rows_in else "MISMATCH"
log_run(READINGS_FILE, rows_in, rows_out, status, "" if status == "SUCCESS" else "rows_in != rows_out")
print(f"hierarchy nodes: {hierarchy.count()}, tags: {tags.count()}, readings in: {rows_in:,}, out: {rows_out:,}, hours: {n_hours}")
if status != "SUCCESS":
    raise ValueError(f"sensor backfill {RUN_ID}: rows_in {rows_in} != rows_out {rows_out}")
